In [0]:
%run ./_common

In [0]:
from pyspark.sql import functions as F

log = get_logger("opspulse.bronze")

# One-time setup: separate volume for streaming checkpoints
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.bronze.checkpoints")
log.info("checkpoint volume ready: %s", CHECKPOINT_ROOT)

In [0]:
def read_landing_stream():
    """Auto Loader: landing zone ki mukammal .json.gz files ko raw text lines ki tarah padho."""
    return (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "text")
        .option("pathGlobFilter", "*.json.gz")
        .load(LANDING_PATH)
        .withColumnRenamed("value", "raw_json")
    )


def add_audit_columns(df):
    """Bronze audit columns, table ke schema ke order mein."""
    return (
        df
        .withColumn("source_file", F.col("_metadata.file_path"))
        .withColumn("source_file_mtime", F.col("_metadata.file_modification_time"))
        .withColumn("ingest_ts", F.current_timestamp())
        .select("raw_json", "source_file", "source_file_mtime", "ingest_ts")
    )

In [0]:
def run_bronze_ingest() -> int:
    """Incremental ingest: processes only files not yet in the checkpoint, then stops.

    Returns the number of newly ingested rows.
    """
    checkpoint = f"{CHECKPOINT_ROOT}/bronze_gh_events_raw"
    before = spark.table(BRONZE_TABLE).count() if spark.catalog.tableExists(BRONZE_TABLE) else 0
    log.info("starting bronze ingest from %s (rows before: %d)", LANDING_PATH, before)
    try:
        query = (
            add_audit_columns(read_landing_stream())
            .writeStream
            .option("checkpointLocation", checkpoint)
            .trigger(availableNow=True)
            .toTable(BRONZE_TABLE)
        )
        query.awaitTermination()
    except Exception as e:
        log.error("bronze ingest failed: %s", e)
        raise
    after = spark.table(BRONZE_TABLE).count()
    log.info("bronze ingest finished: %d new rows, %d total", after - before, after)
    return after - before

In [0]:
run_bronze_ingest()

In [0]:
df = spark.table(BRONZE_TABLE)
print("total rows:", df.count())
display(df.groupBy("source_file").count().orderBy("source_file"))

# Data quality: null or non-JSON-looking rows
invalid = df.filter(F.col("raw_json").isNull() | ~F.col("raw_json").startswith("{")).count()
print("invalid rows:", invalid)

display(
    df.select(F.substring("raw_json", 1, 200).alias("sample"), "source_file", "ingest_ts").limit(3)
)